## BERT Transfer Leanring

Step 1: Define the Dataset (our 10 simple sample tests).

Step 2: Tokenization & Pre-trained BERT Setup.

Step 3: Building and Fine-Tuning the Model.

Step 4: Running Predictions on our Test Cases.

## Step 1: Defining Our 10 Sample Test Cases
For this lab, we'll use a binary sentiment classification task: classifying movie or product reviews as Positive (1) or Negative (0).

In [1]:
import pandas as pd

reviews = [
    "This movie was amazing and inspiring.",
    "Absolutely fantastic acting and storyline.",
    "One of the best films I have ever seen.",
    "Brilliant movie. Highly recommended.",
    "Excellent performances and great direction.",
    "This movie was boring and too long.",
    "Terrible acting and weak storyline.",
    "One of the worst movies ever made.",
    "I regret watching this film.",
    "Awful movie. Complete waste of time."
]

labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

df = pd.DataFrame({
    "review": reviews,
    "label": labels
})

print(df)

                                        review  label
0        This movie was amazing and inspiring.      1
1   Absolutely fantastic acting and storyline.      1
2      One of the best films I have ever seen.      1
3         Brilliant movie. Highly recommended.      1
4  Excellent performances and great direction.      1
5          This movie was boring and too long.      0
6          Terrible acting and weak storyline.      0
7           One of the worst movies ever made.      0
8                 I regret watching this film.      0
9         Awful movie. Complete waste of time.      0


## Step 2: Tokenization & Pretrained BERT Setup

Purpose: Convert text into BERT inputs and load the pretrained BERT tokenizer.

In [2]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

encodings = tokenizer(
    list(df["review"]),
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([10, 12])


## Step 3: Build and Fine-Tune the BERT Model

Purpose: Load a pretrained BERT classifier and train it on our 10 movie reviews.

In [4]:
import torch
from transformers import BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

dataset = TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"],
    torch.tensor(labels)
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

model.train()

for epoch in range(3):

    total_loss = 0

    for batch in loader:

        input_ids, attention_mask, labels_batch = batch

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 Loss: 3.7371
Epoch 2 Loss: 3.2597
Epoch 3 Loss: 2.9945


In [6]:
model.eval()

test_reviews = [
    "The movie was excellent and entertaining.",
    "This film was horrible and boring."
]

inputs = tokenizer(
    test_reviews,
    return_tensors="pt",
    truncation=True,
    padding=True
)

with torch.no_grad():

    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

predictions = torch.argmax(outputs.logits, dim=1)

for review, pred in zip(test_reviews, predictions):

    sentiment = "Positive" if pred.item() == 1 else "Negative"

    print(f"\nReview: {review}")
    print(f"Sentiment: {sentiment}")


Review: The movie was excellent and entertaining.
Sentiment: Positive

Review: This film was horrible and boring.
Sentiment: Negative
